# 02. Modelių mokymas, HP, kalibracija, slenksčiai

> **Peržiūros notebook.** MATLAB fragmentai nevykdomi. Ankstesnis etapas: [01_duomenu_gavimas_ir_paruosimas.ipynb](01_duomenu_gavimas_ir_paruosimas.ipynb).

Mokomi B0 (dauguma), B1 (L2 logistinė regresija), M1 (SVM-RBF), M2 (MLP), M3 (RBF tinklas) ir abliacijos SVM (tiesinis, mean-10, worst-10). Viskas — **tik mokymo 456** + 25 vidiniai skaidiniai.


## Kas čia vyksta paprastai

Kiekvienam modeliui perrenkamas nedidelis hiperparametrų tinklelis. Kiekviena konfigūracija gauna vidutinį ROC-AUC iš 25 foldų (**OOF**). Testas čia nenaudojamas.

Jei dvi konfigūracijos beveik lygios (AUC skiriasi mažiau nei 0,002), imama **paprastesnė** — bet tik **tarp tų, kurios pasiekė beveik maksimumą**, ne „pirma paprasta tinklelyje“.

SVM ir kiti skorai verčiami į tikimybę Platt formule (vėl tik OOF). Tada parenkami slenksčiai `t_cost`, `t_se98` (didžiausias t su OOF Se ≥ 0,98) ir Youden. Jie užrašomi į `models/thresholds.json` ir `freeze.txt`. Testas vis dar užrakintas.


## `models/tune_cv.m` — tinklelio paieška

Kiekviename folde `fit_scaler` + `apply_scaler` + modelio skorai validacijai. `idxTest` tik patikrai. `n==569` → klaida.

```matlab
% models/tune_cv.m (foldo ciklas)
sc = fit_scaler(Xtrain(tr, :), cfg);
Ztr = apply_scaler(Xtrain(tr, :), sc);
Zva = apply_scaler(Xtrain(va, :), sc);
scVa = fold_score(modelName, Ztr, ytrain(tr), Zva, hp, cfg, ...
    cvInner.seed_fold(r, f));
foldAuc(r, f) = roc_auc(ytrain(va), scVa);

% po viso tinklo:
[maxAuc, ~] = max(aucMean);
idxCand = find(aucMean >= maxAuc - 0.002 - 1e-12);
% tarp kandidatų — paprastesnis (is_simpler)
```

HP tinklelis (`config.m`): LR λ ∈ {10⁻⁴…1}; SVM C × KernelScale 7×7; MLP H ∈ {5,10,20}, λ ∈ {0, 0,1, 0,3}, 5 sėklos; RBF J, κ, λ.

Užšaldyti (OOF) HP, naudoti galutiniame fite (iš `tune_cv` / `*.mat`, ne iš testo CSV): LR **λ = 0,1**; SVM-RBF **C = 0,3**, **s = 8**; MLP **H = 5**, **λ = 0,3**; RBF tinklas **J = 40**, **κ = 2**, **λ = 0,1**; tiesinis SVM **C = 0,01** (paprastesnis langelis ±0,002 nuo max AUC).


## B0 — daugumos klasifikatorius (`train_majority.m`)

**Paprastai:** visada spėja gerybinį (B), nes mokyme B daugiau (286 vs 170). Tai apatinė kartelė.

**Techniškai:** pastovi tikimybė ≈ mokymo P(M); slenkstis 0,5. Teste (`main_results.csv`, `majority` / `t_se98`): Se = 0, Sp = 1, FN = 42, AUC = 0,5.


## B1 — L2 logistinė regresija (`train_logreg.m`)

**Paprastai:** tiesė požymių erdvėje: kiekvienas požymis turi svorį, suma per sigmoidę tampa P(M). L2 (ridge) neleidžia svoriams išsipūsti. Klasės svoriai 5:1, nes praleistas piktybinis brangesnis.

**Techniškai:** `fitclinear`, `Learner=logistic`, `Regularization=ridge`. Z jau pagal (1).

```matlab
% models/train_logreg.m (fragmentas)
Mdl = fitclinear(Z, y, ...
    'Learner', 'logistic', ...
    'Regularization', 'ridge', ...
    'Lambda', lambda, ...
    'ClassNames', [0 1], ...
    'Weights', w, ...
    'Solver', 'lbfgs');
```


## M1 — SVM su RBF branduoliu (`train_svm_rbf.m`)

**Paprastai:** ieškoma maksimalios maržos ribos, bet ne tiesės, o lanksčios paviršiaus, kurią apibrėžia Gauso branduolys. Klaidingas negatyvas (praleistas M) kainuoja 5 kartus daugiau nei klaidingas pozityvas.

**Techniškai, Kolokviumo (2)–(3):**

$$
K(z,z')=\exp\big(-\|z-z'\|^2 / s^2\big),\quad \gamma=1/s^2,
$$

$$
f(z)=\sum_{i\in S}\alpha_i y_i K(z_i,z)+b,\quad \hat y=\mathrm{sign}(f(z)).
$$

`Standardize=false`, nes Z jau z-score. `Cost = [0 1; 5 0]`.

```matlab
% models/train_svm_rbf.m (fragmentas)
args = {'KernelFunction', kernel, 'BoxConstraint', C, ...
    'Cost', cfg.costMatrix, 'ClassNames', [0 1], 'Standardize', false};
if ~strcmp(kernel, 'linear')
    args = [args, {'KernelScale', s}];
end
Mdl = fitcsvm(Z, y, args{:});
```


## M2 — MLP (`train_mlp.m`)

**Paprastai:** vienas paslėptas sluoksnis (H neuronų, tanh), išėjimas — sigmoidė P(M). Mokoma 5 skirtingomis sėklomis; `tune_cv` ima vidurkį, kad atsitiktinė inicializacija nespręstų viena „laiminga“ eiga.

**Techniškai:** `patternnet`, `trainscg`. Validacijos poaibis **tik iš foldo mokymo** (`divideind`); `testInd = []`.

```matlab
% models/train_mlp.m (fragmentas)
net = patternnet(H, 'trainscg');
net.performParam.regularization = lambda;
net.divideFcn = 'divideind';
net.divideParam.trainInd = trInd';
net.divideParam.valInd = vaInd';
net.divideParam.testInd = [];
```


## M3 — RBF tinklas (`train_rbfnet.m`)

**Paprastai:** k-means parenka J centrus mokymo Z, aplink juos Gauso kauburėliai, tada ridge regresija į {−1,+1}. Centrai — klasterių vidurkiai, ne SVM palaikymo vektoriai.

**Techniškai, Kolokviumo 3.4:** $w=(\Phi^T\Phi+\lambda I)^{-1}\Phi^T\tilde y$. Nulinis atstumas tarp centrų — klaida, ne tylus pataisymas.

```matlab
% models/train_rbfnet.m (fragmentas)
[~, centers] = kmeans(Z, J, 'Replicates', 5, 'MaxIter', 200);
Phi1 = [ones(n, 1), Phi];
ytilde = 2 * y - 1;
w = (Phi1' * Phi1 + lambda * eye(J + 1)) \ (Phi1' * ytilde);
```


## Platt kalibracija — Kolokviumo (4)

**Paprastai:** SVM `f(z)` nėra tikimybė. Platt pritaiko sigmoidę $P(M|z)=1/(1+e^{A f+B})$ pagal OOF skorų ir žymių poras. Testo balai į `fminunc` **neįleidžiami**.

```matlab
% models/calibrate_platt.m (fragmentas)
t(y == 1) = (Npos + 1) / (Npos + 2);   % t₊
t(y == 0) = 1 / (Nneg + 2);            % t₋
ab = fminunc(@(ab) platt_nll(ab, f, t), x0, opts);
```


## Slenksčiai — Kolokviumo (5)–(6)

**Paprastai:** turint P(M), vis tiek reikia taisyklės „kada sakyti M“. Trys taisyklės fiksuojamos **OOF**, tada aklai taikomos teste.

- $t^*=c_{FP}/(c_{FP}+c_{FN})=1/6$ — Bayes slenkstis kainoms 5:1.
- `t_cost` — tinklelio 0,01 žingsniu tas t, kuris mažina $EC(t)=5\cdot FN+1\cdot FP$.
- `t_se98` — **didžiausias** t, kur OOF Se vis dar ≥ 0,98 (H1 operacinis taškas).
- `t_youden` — max Se+Sp−1.

```matlab
% models/choose_threshold.m (fragmentas)
tgrid = 0:0.01:1;
EC = cFN * FN + cFP * FP;          % (6)
% t_se98 atnaujinamas kiekvieną kartą, kai Se >= cfg.se_target
% → lieka paskutinis (didžiausias) toks t
```

SVM-RBF OOF: `t_cost = 0,33`, `t_se98 = 0,34` — tas pats minimalus EC; tai **ne klaida** (`reports/tables/oof_ec_se_curve.csv`).


## Užšaldymas

`reports/freeze.txt` rašomas **prieš** `evaluate_test`: split 456/113, kaina 5:1, H1 ΔSp riba 0,03, Se tikslas 0,98, 25 CV, SHA-256 modelių `.mat` ir `preregistration.md`. Po to fit/HP/slenksčių keisti negalima.

Grafikai šiame etape dar negeneruojami — jie atsiranda `evaluate_test` (03).

Toliau: [03_vertinimas_ir_hipotezes.ipynb](03_vertinimas_ir_hipotezes.ipynb).
